# Load the 4-bit base and attach LoRA adapters

Stages 1–2 of the SFT pipeline are QLoRA: quantize the frozen base to 4-bit so a 4B model fits in 12 GB, then attach the small trainable LoRA adapters — the only weights we actually train. Build both pieces here and **verify they engaged** before wiring the trainer. If QLoRA isn't really on, every later stage silently trains the wrong thing (full precision → OOM, or no adapters → nothing learns).

## Verify it worked — sanity-check everything
- **trainable % ≈ 0.1–1%.** If it reads ~100%, LoRA didn't attach → bug, stop.
- **`nvidia-smi`: model resident ~3–4 GB**, not ~8 GB. If 8 GB, 4-bit never engaged → bug.
- No dtype/device errors on a one-token forward (optional: `model(**tokenizer("hi", return_tensors="pt").to("cuda"))`).

**Heads-up:** first run downloads ~8 GB of base weights (cached after)

In [ ]:
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import tool_forge

## Configure BitsAndBytes
— `load_in_4bit`, quant type `"nf4"`, double-quant on, compute dtype `bfloat16`.

See https://huggingface.co/blog/4bit-transformers-bitsandbytes

In [ ]:
nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

## Load the base
- `Qwen/Qwen3-4B-Instruct-2507` with that quant config
- `dtype=torch.bfloat16`
- `device_map={"": 0}` (pins all to GPU 0).

On first download, can take several minutes

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    tool_forge.MODEL_NAME,
    quantization_config=nf4_config,
    dtype=torch.bfloat16,
    device_map={"":0}
    )

## Prepare model
Required before LoRA on a quantized base (fp32 LayerNorms, grad-checkpoint compat, embeds `requires_grad`).

In [ ]:
model = prepare_model_for_kbit_training(model)

## Configure LoRa
- `r=16`
- `lora_alpha=32`
- `target_modules="all-linear"`
- `lora_dropout=0.05`
- `task_type="CAUSAL_LM"`.

See: https://huggingface.co/docs/peft/tutorial/peft_model_config

*Notes:*
- Hueristic for `alpha` = 2 x `r`
- Bigger `r` gives more capacity at linear memory costs. `r x (in + out)` 
- Memory is bigger constraint than capacity for this

In [ ]:
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16,
    target_modules="all-linear",
    lora_alpha=32,
    lora_dropout=0.05,
    )

## Apply LoRa adapters

In [ ]:
model = get_peft_model(model, lora_config)
type(model).__name__

## Outcome
The **trainable-param numbers** and the **VRAM footprint** — those two confirm QLoRA is correctly engaged.

### Model parameters
To determine if the model was correctly quantized, we can calculate the expected trainable params based on our `r=16` parameter.

- q 16 x (2560 + 4096) = **106,496** 
- o 16 x (4096 + 2560) = **106,496**
- k 16 x (2560 + 1024) = **57,344**
- gate/up/down 16 x (2560 + 9728) = **196,608 x3**  
- v 16 x (2560 + 1024) = **57,344** 
- **917,504**/block × 36 blocks = **33,030,144**

In [ ]:
model.print_trainable_parameters()

#### Result
The trainable % of params is 0.8145% of the total params.

### Model GPU footprint

In [ ]:
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

#### Expected vs actual
`nvidia-smi` reports *everything* on the GPU — our model **plus** the CUDA context, PyTorch's reserved pool, and (on WSL) the Windows display — so it overcounts the model badly. Instead, compute what the model *should* weigh from how each tensor is stored, then compare to `torch.cuda.memory_allocated()`, which counts only our tensors.

Storage per bucket:
- **embeddings + output head** — NOT quantized → bf16, `2 B/param`
- **transformer projections** — 4-bit → `~0.5 B/param` (+ double-quant scales)
- **LoRA adapters** — bf16, `2 B/param`

In [ ]:
GB = 1e9
V, H = model.config.vocab_size, model.config.hidden_size   # 151936, 2560
tied = model.config.tie_word_embeddings                    # 4B is untied -> head counts separately
untied = 1 if tied else 2

embed_head = untied * V * H                  # token embeddings (+ output head), bf16
base_total = 4_055_498_240 - 33_030_144      # all base params (logical), minus LoRA adapters
nf4        = base_total - embed_head         # everything else -> 4-bit

gb_embed = embed_head * 2   / GB             # bf16 = 2 B/param
gb_nf4   = nf4 * 0.5        / GB             # 4-bit = 0.5 B/param
gb_scale = nf4 * (1/64)     / GB             # double-quant: ~1 byte absmax / 64 weights
gb_lora  = 33_030_144 * 2   / GB             # adapters, bf16

expected = gb_embed + gb_nf4 + gb_scale + gb_lora
print(f"embed+head (bf16): {gb_embed:5.2f} GB   (tied={tied})")
print(f"4-bit blocks:      {gb_nf4:5.2f} GB")
print(f"quant scales:      {gb_scale:5.2f} GB")
print(f"LoRA adapters:     {gb_lora:5.2f} GB")
print(f"--> expected model:{expected:5.2f} GB")

In [ ]:
# Tensors should land near `expected`; the rest of nvidia-smi is overhead + display.
print(f"allocated (our tensors): {torch.cuda.memory_allocated()/GB:.2f} GB")
print(f"reserved  (torch pool):  {torch.cuda.memory_reserved()/GB:.2f} GB")
# nvidia-smi 'used' - reserved  ~=  CUDA context + WSL display + other processes

### Test sample output

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(tool_forge.MODEL_NAME)
model(**tokenizer("hi", return_tensors="pt").to("cuda"))